## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. You can read every line of the code below before you run it.

# Share price and store visitors: unit root test

**Unit Root Test** · a worked example from the StatNex teaching guide

**The story.** Fifty days of two series: the share price of a listed company, and the number of visitors to one of its stores. The share price wanders up and down with no fixed level, while store visits bounce around and keep being pulled back towards their usual range.

**The question.** Which of the two series returns to a stable path, and which one wanders without limit?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `08_time_series/unit_root_test.csv` (50 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Day,Stock price ($),Store visitors
1.0,53.17,324.0
2.0,51.86,339.0
3.0,52.19,328.0
4.0,53.06,342.0
5.0,52.72,298.0
6.0,52.97,340.0
7.0,51.37,342.0
8.0,50.68,330.0
9.0,50.75,313.0
10.0,49.96,338.0
11.0,49.06,352.0
12.0,46.7,329.0
13.0,46.39,295.0
14.0,46.15,313.0
15.0,43.9,301.0
16.0,42.47,348.0
17.0,43.4,332.0
18.0,44.23,335.0
19.0,44.24,386.0
20.0,44.38,372.0
21.0,43.46,369.0
22.0,45.15,358.0
23.0,45.62,341.0
24.0,45.21,311.0
25.0,45.64,305.0
26.0,46.74,342.0
27.0,46.87,313.0
28.0,47.59,320.0
29.0,47.01,304.0
30.0,46.92,308.0
31.0,48.39,336.0
32.0,47.23,338.0
33.0,49.27,299.0
34.0,50.15,297.0
35.0,51.1,278.0
36.0,51.14,314.0
37.0,52.87,327.0
38.0,52.9,323.0
39.0,51.5,344.0
40.0,51.92,325.0
41.0,51.76,297.0
42.0,52.35,300.0
43.0,51.88,358.0
44.0,50.49,322.0
45.0,51.8,314.0
46.0,51.55,309.0
47.0,52.59,290.0
48.0,50.78,285.0
49.0,49.68,281.0
50.0,50.79,276.0
""")

## Helper: unit_root_test

A step that no library does, written out. In the code download it is the file `unit_root_test.py`, which the analysis imports; here the analysis uses it directly.

In [ ]:
"""Two P values for the unit root test that statsmodels does not give.

dickey_fuller_p: statsmodels' adfuller() reports MacKinnon's response-surface P value.
This one interpolates the printed Dickey-Fuller table instead (constant and trend;
Banerjee et al. 1993, Table 4.2), as R's tseries::adf.test() does. The table spans 1% to
99% only, so a statistic beyond either end is reported as "below 0.01" / "above 0.99".

durbin_watson_p: statsmodels' durbin_watson() gives the ratio but no P value. The ratio's
null distribution depends on the design. By default the two-sided P value comes from that
exact distribution (Imhof's integral; R's lmtest::dwtest(exact = TRUE)); exact=False gives
a normal approximation on its exact mean and variance (lmtest::dwtest(exact = FALSE)).
"""

import numpy as np
from scipy import integrate, stats

#: Sample sizes (rows) and probabilities (columns) of the table.
SIZES = [25, 50, 100, 250, 500, 100000]
PROBABILITIES = [0.01, 0.025, 0.05, 0.10, 0.90, 0.95, 0.975, 0.99]
TABLE = np.array([
    [-4.38, -4.15, -4.04, -3.99, -3.98, -3.96],
    [-3.95, -3.80, -3.73, -3.69, -3.68, -3.66],
    [-3.60, -3.50, -3.45, -3.43, -3.42, -3.41],
    [-3.24, -3.18, -3.15, -3.13, -3.13, -3.12],
    [-1.14, -1.19, -1.22, -1.23, -1.24, -1.25],
    [-0.80, -0.87, -0.90, -0.92, -0.93, -0.94],
    [-0.50, -0.58, -0.62, -0.64, -0.65, -0.66],
    [-0.15, -0.24, -0.28, -0.31, -0.32, -0.33],
])


def dickey_fuller_p(statistic, n):
    """(P, note): the table's critical values interpolated to n, then P read off them.

    n is the number of differences, len(series) - 1. np.interp holds the end values
    outside the table, so P stops at 0.01 / 0.99 and the note says it went past.
    """
    critical = [np.interp(n, SIZES, row) for row in TABLE]
    p = float(np.interp(statistic, critical, PROBABILITIES))
    note = "below 0.01" if statistic < critical[0] else "above 0.99" if statistic > critical[-1] else ""
    return p, note


def durbin_watson_p(residual, design, exact=True):
    """Two-sided P of the Durbin-Watson ratio.

    A is the second-difference matrix, so that DW = e'Ae / e'e.

    exact=True: under H0 the residuals are M times independent normal errors. With Z an
    orthonormal basis of the residual space, DW <= c exactly when sum((nu_i - c) w_i^2)
    <= 0, where nu_i are the eigenvalues of Z'AZ and w_i standard normals; Imhof's (1961)
    integral gives that probability, and the P value is twice the smaller tail. As the
    app does, a regression with more than 1,000 residuals uses the approximation below.

    exact=False: with M = I - X (X'X)^-1 X' the residual maker, the mean is
    tr(MA) / (n - k) and the variance 2 [tr((MA)^2) - (n - k) mean^2] / ((n - k)(n - k + 2)),
    and a normal distribution with those two moments gives the P value.

    Both build n x n matrices, which is fine for a few thousand rows.
    """
    n, k = design.shape
    d = np.sum(np.diff(residual) ** 2) / np.sum(residual ** 2)
    A = 2 * np.eye(n) - np.eye(n, k=1) - np.eye(n, k=-1)
    A[0, 0] = A[-1, -1] = 1
    if exact and n <= 1000:
        Z = np.linalg.qr(design, mode="complete")[0][:, k:]
        weights = np.linalg.eigvalsh(Z.T @ A @ Z) - d

        def integrand(u):
            theta = 0.5 * np.sum(np.arctan(weights * u))
            log_rho = 0.25 * np.sum(np.log1p((weights * u) ** 2))
            return np.sin(theta) * np.exp(-log_rho) / u

        integral, _ = integrate.quad(integrand, 0, np.inf, limit=500, epsabs=1e-13, epsrel=1e-11)
        below = min(max(0.5 - integral / np.pi, 0.0), 1.0)
        return 2 * min(below, 1 - below)
    M = np.eye(n) - design @ np.linalg.pinv(design)
    MA = M @ A
    mean = np.trace(MA) / (n - k)
    variance = 2 * (np.trace(MA @ MA) - (n - k) * mean ** 2) / ((n - k) * (n - k + 2))
    return 2 * stats.norm.sf(abs(d - mean) / np.sqrt(variance))

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.stats.stattools import durbin_watson
from statsmodels.tsa.stattools import adfuller
# dickey_fuller_p and durbin_watson_p: defined in the helper cell unit_root_test above (unit_root_test.py in the code download)

pd.set_option("display.width", 120)

## The data

In [ ]:
# A time series in row order. The order of the rows is the data, so a series may not
# have blank cells.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["Stock price ($)", "Store visitors"]
# Each column is tested as its own series.
series = {name: data[name] for name in columns}
lags = [0]
# The Durbin-Watson P value from the ratio's exact distribution (see unit_root_test.py).
exact_dw = True

## Augmented Dickey-Fuller test (constant and trend)

In [ ]:
# Does the series keep returning to a path, or wander off for good? The test regresses
# each period's change dx(t) on a constant, a linear trend, the previous level x(t-1)
# and `lag` earlier changes. A negative coefficient on x(t-1) means a high level is
# followed by a fall: the series is pulled back. H0: that coefficient is 0, a unit
# root (a random walk, in which a shock never fades). The statistic is its t value,
# which under H0 does not follow the t distribution, so its P value comes from the
# Dickey-Fuller table (see unit_root_test.py), as R's tseries::adf.test() reads it,
# not from the MacKinnon approximation adfuller() reports (near 1% the two can differ
# by a factor of two). A small P is evidence against a unit root: with the trend in
# the regression, the series returns to a straight-line path, which may slope. A
# large P means the test could not tell the series from a random walk, not that it
# is one.
# Durbin-Watson: near 2 means the regression's errors show no pattern from one period
# to the next, which the test assumes; if they do, add lags. It is a first check only:
# with x(t-1) among the regressors it leans towards 2. AIC = rows * log(RSS / rows)
# + 2k compares the lag orders of one series (smaller is better).
sizes, tests = [], []
for name, values in series.items():
    x = values.to_numpy(float)
    sizes.append(len(x))
    rows = []
    for lag in lags:
        statistic, _, _, store = adfuller(x, maxlag=lag, regression="ct", autolag=None,
                                          regresults=True)
        fit = store.resols
        p, note = dickey_fuller_p(statistic, len(x) - 1)
        aic = fit.nobs * np.log(fit.ssr / fit.nobs) + 2 * len(fit.params)
        rows.append([lag, durbin_watson(fit.resid), durbin_watson_p(fit.resid, fit.model.exog, exact_dw),
                     aic, statistic, p, note])
    test = pd.DataFrame(rows, columns=["Lag", "Durbin-Watson", "P (DW)", "AIC",
                                       "Dickey-Fuller", "P", ""])
    tests.append(test)
    print(f"\n{name}: n = {len(x)}")
    print(test.to_string(index=False))

## What the result says

A series has a unit root when it wanders with no level to return to. The Dickey-Fuller test starts from the idea that the series wanders, and looks for evidence of a pull back to a steady path. A small P value means such a pull was found.

For store visitors the Dickey-Fuller statistic is -4.14 and the P value is 0.011, so the pull is there. For the share price the statistic is -1.78 and the P value is 0.661: no pull is found. The Durbin-Watson ratios, 1.98 and 1.95, are close to 2, which means the test's errors show no leftover pattern.

Store visitors keep coming back to a steady path, so their usual range is a fair guide for planning. For the share price the test only fails to find a pull. 50 days are too few to prove that it just wanders.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.